In [1]:
import logging
import tqdm
import random
from exp.run import ExperimentRun
from exp.config import TransformerExperiments, CNNExperiments, LargeTransformerExperiments
logging.basicConfig(level=logging.ERROR)

In [2]:
configures = [
	CNNExperiments(),
	TransformerExperiments(),
	LargeTransformerExperiments()
]
config = configures[-1]
total_run: int = 600
entire_experiments: bool = True

In [ ]:
if entire_experiments:
	# Prepare all Executor instance
	instances = []
	for conf in configures:
		conf.debug = False
		conf.repeats = 5
		conf.gpu_id = 1
		exp = ExperimentRun(config=conf)
		instances.append(exp)

	print(f"Preparing Monte Carlo Data...")
	for i in tqdm.tqdm(range(total_run)):
		instance: ExperimentRun = random.choice(instances)
		instance.prepare_monte_carlo_experiments_data(number=1)

	# Executing all experiments
	for instance in instances:
		print(f"Executing {instance._config.run_id}...")
		if isinstance(instance._config, LargeTransformerExperiments):
			instance.run_experiments_solving_compatibility_issue()
		else:
			instance.run_experiments()
		instance.to_evaluation_result()
else:
	exp = ExperimentRun(config=config)
	exp.run_monte_carlo_experiments(total_run)
	exp.to_evaluation_result()


